In [35]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, size, dayofweek, hour, to_date, count, avg
import pyspark.sql.functions as f 


spark = SparkSession.builder.appName("twitter_transformation_23").getOrCreate()

df_silver = spark.read.parquet('/home/guiandreis/Airflow-tweets-spark-docker/airflow-spark-teste/data/silver')


df_silver = df_silver.withColumn(
    "engagement",
    col("likes") +
    col("retweets") +
    col("replies") +
    col("quotes")
)



gold_df = df_silver.withColumn("num_edits", size(col("edit_versions_ids")))
    
    
avg_eng_per_day = gold_df.groupBy("created_date") \
    .agg(avg("engagement").alias("avg_eng_per_day"))

gold_df = gold_df.join(avg_eng_per_day, on="created_date", how="left")


user_tweet_counts = gold_df.groupBy("user_id") \
    .agg(
        count("*").alias("total_tweets"),
        avg("engagement").alias("avg_eng_per_user")
    )

gold_df = gold_df.join(user_tweet_counts, on="user_id", how="left")


    
gold_df = gold_df.select(
    "user_id", "thread_id", "tweet_id", "created_at", "created_date",
    "tweet_text","likes", "quotes", "replies", "retweets", "engagement", 
    "avg_eng_per_day", "total_tweets", "avg_eng_per_user",
    "language")


gold_df.show()
 #% de replies vs tweets 

+-------+---------+--------+--------------------+------------+--------------------+-----+------+-------+--------+----------+-----------------+------------+------------------+--------+
|user_id|thread_id|tweet_id|          created_at|created_date|          tweet_text|likes|quotes|replies|retweets|engagement|  avg_eng_per_day|total_tweets|  avg_eng_per_user|language|
+-------+---------+--------+--------------------+------------+--------------------+-----+------+-------+--------+----------+-----------------+------------+------------------+--------+
|     26|       30|       5|2026-02-02T11:36:...|  2026-02-02|Outro tweet fictí...|   71|    33|     24|      90|       218|197.7888888888889|           2|             178.0|      en|
|    100|        7|      45|2026-02-02T11:42:...|  2026-02-02|Este é um tweet f...|   53|    69|     76|      40|       238|197.7888888888889|           6|134.16666666666666|      en|
|     18|       26|       3|2026-02-02T09:23:...|  2026-02-02|Outro tweet fictí.